# Categorical Feature Exploration

**Datasets:** `freMTPL2freq` and `bemtpl` from CASdatasets (R package)  
**Purpose:** Inspect the distribution and mean claim rate for every categorical and ordinal
feature to inform encoding decisions. No preprocessing is applied — data is loaded raw from R.

**How to use:**
1. Run all cells top to bottom.
2. For each feature: check the value counts table and the bar chart.
3. Record your encoding decisions in the **Encoding decisions** cell at the bottom.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display, Markdown

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr

pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", "{:.2f}".format)
plt.rcParams.update({
    "figure.dpi": 100,
    "axes.titlesize": 12,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

In [ ]:
def load_casdataset(name: str) -> pd.DataFrame:
    """Load a named dataset from the CASdatasets R package via rpy2."""
    try:
        importr("CASdatasets")
    except Exception as e:
        raise ImportError(
            "CASdatasets R package not found. "
            "See environment.txt Step 4 for installation instructions."
        ) from e
    ro.r(f"data({name})")
    with localconverter(ro.default_converter + pandas2ri.converter):
        df = ro.conversion.rpy2py(ro.r[name])
    # R factors arrive as pandas Categorical -- convert to plain strings
    for col in df.select_dtypes("category").columns:
        df[col] = df[col].astype(str)
    return df


print("Loading freMTPL2freq ...")
df_freq = load_casdataset("freMTPL2freq")
print(f"  {df_freq.shape[0]:,} rows x {df_freq.shape[1]} columns")

print("Loading freMTPL2sev ...")
df_sev = load_casdataset("freMTPL2sev")
print(f"  {df_sev.shape[0]:,} rows x {df_sev.shape[1]} columns")

print("Loading beMTPL97 ...")
df_bemtpl = load_casdataset("beMTPL97")
print(f"  {df_bemtpl.shape[0]:,} rows x {df_bemtpl.shape[1]} columns")

In [ ]:
def explore_feature(
    df: pd.DataFrame,
    col: str,
    response_col: str,
    exposure_col: str,
    order,
    figsize_base: tuple = (10, 4),
) -> None:
    """
    Display value counts, mean claim-rate table, and bar chart for one feature.

    Parameters
    ----------
    order : 'rate' | 'natural' | list
        'rate'    - sort levels by mean claim rate, descending.
        'natural' - sort levels by their own value, ascending (numeric ordinals).
        list      - explicit ordering, e.g. ['TPL', 'TPL+', 'TPL++'].
    """
    n_levels = df[col].nunique()

    # -- Value counts --------------------------------------------------------
    counts = (
        df[col]
        .value_counts(dropna=False)
        .rename_axis(col)
        .reset_index(name="count")
    )
    counts["pct"] = (counts["count"] / len(df) * 100).round(2)

    # -- Aggregate claims and exposure per level ------------------------------
    agg = (
        df.groupby(col, observed=True)
        .agg(
            n_policies=(response_col, "count"),
            sum_claims=(response_col, "sum"),
            sum_exposure=(exposure_col, "sum"),
        )
        .reset_index()
    )
    agg["mean_rate"] = agg["sum_claims"] / agg["sum_exposure"]

    summary = agg.merge(counts[[col, "count", "pct"]], on=col)

    # -- Apply ordering ------------------------------------------------------
    if order == "rate":
        summary = summary.sort_values("mean_rate", ascending=False)
    elif order == "natural":
        try:
            summary[col] = pd.to_numeric(summary[col])
        except (ValueError, TypeError):
            pass
        summary = summary.sort_values(col)
    else:  # explicit list
        summary[col] = pd.Categorical(summary[col], categories=order, ordered=True)
        summary = summary.sort_values(col)

    x_labels = summary[col].astype(str).tolist()

    # -- Header and summary table --------------------------------------------
    display(Markdown("---"))
    display(Markdown(f"### `{col}` &nbsp; ({n_levels} levels)"))
    display(
        summary[
            [col, "count", "pct", "n_policies", "sum_claims", "sum_exposure", "mean_rate"]
        ]
        .reset_index(drop=True)
        .style.format(
            {
                "count": "{:,.0f}",
                "pct": "{:.2f}%",
                "n_policies": "{:,.0f}",
                "sum_claims": "{:,.2f}",
                "sum_exposure": "{:,.2f}",
                "mean_rate": "{:.2f}",
            }
        )
    )

    # -- Bar chart -----------------------------------------------------------
    # Cap width at 28 inches; wide charts can be scrolled horizontally in JupyterLab.
    fig_w = min(max(figsize_base[0], n_levels * 0.40), 28)
    fig, ax = plt.subplots(figsize=(fig_w, figsize_base[1]))
    ax.bar(
        range(n_levels),
        summary["mean_rate"].tolist(),
        color="steelblue",
        edgecolor="white",
        linewidth=0.5,
    )
    ax.set_xticks(range(n_levels))
    rotation = 90 if n_levels > 15 else (45 if n_levels > 6 else 0)
    ha = "right" if 0 < rotation < 90 else "center"
    ax.set_xticklabels(x_labels, rotation=rotation, ha=ha, fontsize=9)
    ax.set_xlabel(col, fontsize=11)
    ax.set_ylabel("Mean claim rate", fontsize=10)
    ax.set_title(f"Mean claim rate by {col}", fontsize=12)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
    plt.tight_layout()
    plt.show()

---
## 1. French MTPL &mdash; `freMTPL2freq`

**Response:** `ClaimNb` (number of claims per policy)  
**Exposure:** `Exposure` (policy-years)  
**Claim rate:** `sum(ClaimNb) / sum(Exposure)` claims per exposure-year

In [ ]:
print(f"Shape: {df_freq.shape[0]:,} rows x {df_freq.shape[1]} columns")
print()
print("Column dtypes:")
print(df_freq.dtypes.to_string())
print()
overall_rate = df_freq["ClaimNb"].sum() / df_freq["Exposure"].sum()
pct_with_claim = (df_freq["ClaimNb"] > 0).mean()
print(f"Overall claim rate : {overall_rate:.6f} claims / exposure-year")
print(f"Policies with claim: {pct_with_claim:.4%}")
print()
display(df_freq.head())

In [ ]:
# Feature configuration for freMTPL2freq
#
# order:
#   list      -> explicit level ordering  (Area: A < B < C < D < E < F)
#   'natural' -> ascending by value       (numeric ordinals: VehPower, VehAge, etc.)
#   'rate'    -> descending by claim rate (nominals without natural order: VehBrand, Region)

FREQ_FEATURES = [
    {"col": "Area",       "order": ["A", "B", "C", "D", "E", "F"]},
    {"col": "VehPower",   "order": "natural"},
    {"col": "VehAge",     "order": "natural"},
    {"col": "DrivAge",    "order": "natural"},
    {"col": "BonusMalus", "order": "natural"},
    {"col": "VehBrand",   "order": "rate"},
    {"col": "VehGas",     "order": "rate"},
    {"col": "Region",     "order": "rate"},
]

for feat in FREQ_FEATURES:
    explore_feature(
        df=df_freq,
        col=feat["col"],
        response_col="ClaimNb",
        exposure_col="Exposure",
        order=feat["order"],
    )

---
## 2. French MTPL &mdash; `freMTPL2sev`

**Response:** `ClaimNb` (number of claims per policy)  
**Exposure:** `Exposure` (policy-years)  
**Claim rate:** `sum(ClaimNb) / sum(Exposure)` claims per exposure-year

In [ ]:
print(f"Shape: {df_sev.shape[0]:,} rows x {df_sev.shape[1]} columns")
print()
print("Column dtypes:")
print(df_sev.dtypes.to_string())
display(df_sev.head())

---
## 3. Belgian MTPL &mdash; `bemtpl`

**Response:** `nclaims` (number of claims per policy)  
**Exposure:** `expo` (policy-years)  
**Claim rate:** `sum(nclaims) / sum(expo)` claims per exposure-year

> `bemtpl` is used for both frequency and severity modelling (Q1 and Q2).  
> This section explores claim **frequency** for all features.

In [ ]:
print(f"Shape: {df_bemtpl.shape[0]:,} rows x {df_bemtpl.shape[1]} columns")
print()
print("Column dtypes:")
print(df_bemtpl.dtypes.to_string())
print()
overall_rate_be = df_bemtpl["nclaims"].sum() / df_bemtpl["expo"].sum()
pct_with_claim_be = (df_bemtpl["nclaims"] > 0).mean()
print(f"Overall claim rate : {overall_rate_be:.6f} claims / exposure-year")
print(f"Policies with claim: {pct_with_claim_be:.4%}")
print()
display(df_bemtpl.head())

In [ ]:
display(df_bemtpl[(df_bemtpl["postcode"] == 9700)].head().round(2))

In [ ]:
# Feature configuration for bemtpl
#
# coverage has a natural order: TPL < TPL+ < TPL++ (increasing coverage breadth).
# Numeric ordinals (bm, power, agec, ageph) are sorted by their own value.
# Nominal categoricals (fuel, sex, use) are sorted by mean claim rate.
#
# Note: column names are verified from the dtypes printout in the cell above.
# Update this list if the actual dataset uses different column names.

BEMTPL_FEATURES = [
    {"col": "coverage", "order": ["TPL", "TPL+", "TPL++"]},
    {"col": "fuel",     "order": "rate"},
    {"col": "sex",      "order": "rate"},
    {"col": "use",      "order": "rate"},
    {"col": "fleet",    "order": "natural"},
    {"col": "bm",       "order": "natural"},
    {"col": "power",    "order": "natural"},
    {"col": "agec",     "order": "natural"},
    {"col": "ageph",    "order": "natural"},
]

for feat in BEMTPL_FEATURES:
    explore_feature(
        df=df_bemtpl,
        col=feat["col"],
        response_col="nclaims",
        exposure_col="expo",
        order=feat["order"],
    )

## Encoding decisions

> Decisions recorded here will be hard-coded in `config/features.yaml`.

**Type key**
- *ordinal string* — string levels with a meaningful natural order
- *ordinal integer* — integer already on the right scale; pass through as numeric
- *nominal string* — string levels with no natural order
- *nominal string (binary)* — two-level nominal; one-hot and label encoding are equivalent
- *binary string* — stored as `"0"`/`"1"` strings; convert to int before any encoding

| Feature | Dataset | Type | GLM | XGBoost / TabPFN | Notes |
|---|---|---|---|---|---|
| `Area` | freMTPL2freq | ordinal string | one-hot | ordinal encoding | A=0, B=1, C=2, D=3, E=4, F=5 |
| `VehPower` | freMTPL2freq | ordinal integer | as-is (numeric) | as-is (numeric) | |
| `VehAge` | freMTPL2freq | ordinal integer | as-is (numeric) | as-is (numeric) | |
| `DrivAge` | freMTPL2freq | ordinal integer | as-is (numeric) | as-is (numeric) | |
| `BonusMalus` | freMTPL2freq | ordinal integer | as-is (numeric) | as-is (numeric) | |
| `VehBrand` | freMTPL2freq | nominal string | one-hot | label encoding | |
| `VehGas` | freMTPL2freq | nominal string (binary) | binary (0/1) | binary (0/1) | Diesel=0, Regular=1 |
| `Region` | freMTPL2freq | nominal string | one-hot | label encoding | |
| `coverage` | beMTPL97 | ordinal string | one-hot | ordinal encoding | TPL=0, TPL+=1, TPL++=2 |
| `sex` | beMTPL97 | nominal string (binary) | binary (0/1) | binary (0/1) | female=0, male=1 |
| `fuel` | beMTPL97 | nominal string (binary) | binary (0/1) | binary (0/1) | diesel=0, gasoline=1 |
| `use` | beMTPL97 | nominal string | one-hot | label encoding | Levels: private / work / both |
| `fleet` | beMTPL97 | binary string | binary (0/1) | binary (0/1) | Cast str → int directly |
| `bm` | beMTPL97 | ordinal integer | as-is (numeric) | as-is (numeric) | Bonus-malus coefficient |
| `power` | beMTPL97 | ordinal integer | as-is (numeric) | as-is (numeric) | Vehicle horsepower |
| `agec` | beMTPL97 | ordinal integer | as-is (numeric) | as-is (numeric) | Vehicle age in years |
| `ageph` | beMTPL97 | ordinal integer | as-is (numeric) | as-is (numeric) | Policyholder age in years |